# 🏥 KiTS23 Kidney Tumor Segmentation with nnU-Net

Complete pipeline for 3D kidney and tumor segmentation using state-of-the-art nnU-Net.

**Dataset**: KiTS23 (599 cases, 3 classes: Kidney, Tumor, Cyst)  
**Model**: nnU-Net 3D Full Resolution (Won KiTS19 with 91.23% Dice)

## 1. Environment Setup

In [ ]:
# Install required packages (run once)
# !pip install nnunetv2 nibabel SimpleITK pandas tqdm matplotlib kits23

In [ ]:
import os
import json
import shutil
from pathlib import Path
from tqdm import tqdm
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt

# ===== CONFIGURATION =====
KITS23_DIR = Path("./kits23")  # Your KiTS23 dataset directory
NNUNET_RAW = Path("./nnUNet_raw")
NNUNET_PREPROCESSED = Path("./nnUNet_preprocessed")
NNUNET_RESULTS = Path("./nnUNet_results")

DATASET_ID = 23
DATASET_NAME = f"Dataset{DATASET_ID:03d}_KiTS23"

# Create directories
for d in [NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS]:
    d.mkdir(exist_ok=True, parents=True)

# Set nnU-Net environment variables
os.environ['nnUNet_raw'] = str(NNUNET_RAW.absolute())
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED.absolute())
os.environ['nnUNet_results'] = str(NNUNET_RESULTS.absolute())

print("✅ Environment configured")
print(f"   KiTS23: {KITS23_DIR.absolute()}")

## 2. Download Imaging Data (Required!)

⚠️ **Your dataset currently only has segmentation masks. Imaging data must be downloaded separately.**

In [ ]:
# Check current dataset status
available_cases = sorted([d.name for d in KITS23_DIR.iterdir() if d.is_dir() and d.name.startswith('case_')])
print(f"Found {len(available_cases)} case folders")

# Check if imaging exists
sample_case = KITS23_DIR / available_cases[0]
has_imaging = (sample_case / "imaging.nii.gz").exists()
has_segmentation = (sample_case / "segmentation.nii.gz").exists()

print(f"\nData status for {available_cases[0]}:")
print(f"  ✅ Segmentation: {'Found' if has_segmentation else 'Missing'}")
print(f"  {'✅' if has_imaging else '❌'} Imaging: {'Found' if has_imaging else 'MISSING - Need to download!'}")

In [ ]:
# Download imaging data using official kits23 package
if not has_imaging:
    print("📥 Downloading imaging data...")
    print("   This may take a while (~54GB for full dataset)")
    print("")
    
    # Option 1: Using kits23 package (recommended)
    try:
        from kits23 import kits23_download
        # Download to current kits23 directory
        # kits23_download()  # Uncomment to run
        print("Run: kits23_download() or use terminal command below")
    except ImportError:
        print("Install kits23 package first: pip install kits23")
    
    print("")
    print("Alternative: Run in terminal:")
    print("  pip install kits23")
    print("  kits23_download")
else:
    print("✅ Imaging data already available!")

## 3. Calculate Lesion Sizes (Before Training)

In [ ]:
# Method 1: From clinical metadata (instant - no model needed)
with open(KITS23_DIR / "kits23.json", 'r') as f:
    clinical_data = json.load(f)

lesion_info = []
for case in clinical_data:
    if case['case_id'] in available_cases:
        lesion_info.append({
            'case_id': case['case_id'],
            'radiographic_size_cm': case.get('radiographic_size'),
            'pathologic_size_cm': case.get('pathologic_size'),
            'tumor_subtype': case.get('tumor_histologic_subtype'),
            'malignant': case.get('malignant')
        })

lesion_df = pd.DataFrame(lesion_info)
print("📏 Lesion Sizes from Clinical Metadata:")
display(lesion_df)

In [ ]:
# Method 2: From segmentation masks (actual voxel-based volume)
def calculate_volumes(seg_path):
    """Calculate lesion volumes from segmentation mask"""
    seg = nib.load(seg_path)
    data = seg.get_fdata()
    voxel_vol = np.prod(seg.header.get_zooms())  # mm³
    
    return {
        'kidney_cm3': (np.sum(data == 1) * voxel_vol) / 1000,
        'tumor_cm3': (np.sum(data == 2) * voxel_vol) / 1000,
        'cyst_cm3': (np.sum(data == 3) * voxel_vol) / 1000,
        'shape': data.shape,
        'spacing_mm': seg.header.get_zooms()
    }

print("📊 Calculating volumes from segmentation masks...")
volumes = []
for case_id in tqdm(available_cases[:5]):  # First 5 for quick test
    seg_path = KITS23_DIR / case_id / "segmentation.nii.gz"
    if seg_path.exists():
        vol = calculate_volumes(seg_path)
        vol['case_id'] = case_id
        volumes.append(vol)

vol_df = pd.DataFrame(volumes)
print("\n📏 Lesion Volumes from Masks:")
display(vol_df[['case_id', 'kidney_cm3', 'tumor_cm3', 'cyst_cm3']])

## 4. Convert to nnU-Net Format

In [ ]:
def convert_kits23_to_nnunet():
    """Convert KiTS23 to nnU-Net format"""
    dataset_dir = NNUNET_RAW / DATASET_NAME
    images_tr = dataset_dir / "imagesTr"
    labels_tr = dataset_dir / "labelsTr"
    
    images_tr.mkdir(exist_ok=True, parents=True)
    labels_tr.mkdir(exist_ok=True, parents=True)
    
    cases = sorted([d for d in KITS23_DIR.iterdir() 
                   if d.is_dir() and d.name.startswith('case_')])
    
    training_cases = []
    skipped = []
    
    for case_dir in tqdm(cases, desc="Converting"):
        case_id = case_dir.name
        img_path = case_dir / "imaging.nii.gz"
        seg_path = case_dir / "segmentation.nii.gz"
        
        if not img_path.exists():
            skipped.append(case_id)
            continue
            
        if not seg_path.exists():
            skipped.append(case_id)
            continue
        
        # nnU-Net format: case_XXXXX_0000.nii.gz for images
        dst_img = images_tr / f"{case_id}_0000.nii.gz"
        dst_seg = labels_tr / f"{case_id}.nii.gz"
        
        if not dst_img.exists():
            shutil.copy2(img_path, dst_img)
        if not dst_seg.exists():
            shutil.copy2(seg_path, dst_seg)
        
        training_cases.append(case_id)
    
    # Create dataset.json
    dataset_json = {
        "channel_names": {"0": "CT"},
        "labels": {"background": 0, "kidney": 1, "tumor": 2, "cyst": 3},
        "numTraining": len(training_cases),
        "file_ending": ".nii.gz",
        "overwrite_image_reader_writer": "SimpleITKIO"
    }
    
    with open(dataset_dir / "dataset.json", 'w') as f:
        json.dump(dataset_json, f, indent=4)
    
    print(f"\n✅ Converted {len(training_cases)} cases")
    if skipped:
        print(f"⚠️ Skipped {len(skipped)} cases (missing imaging)")
    
    return dataset_dir, training_cases

# Check if imaging is available before converting
if has_imaging:
    dataset_dir, training_cases = convert_kits23_to_nnunet()
else:
    print("❌ Cannot convert - imaging data not available")
    print("   Please run the download step first (Section 2)")

## 5. Preprocessing & Training

In [ ]:
# Step 1: Plan and preprocess
print("="*50)
print("STEP 1: PREPROCESSING")
print("="*50)
print(f"\nRun in terminal:")
print(f"  nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity")
print("\n⏱️ This takes 30-60 minutes")

In [ ]:
# Uncomment to run preprocessing from notebook
# !nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity

In [ ]:
# Step 2: Training
print("="*50)
print("STEP 2: TRAINING")
print("="*50)

FOLD = 0  # 0-4 for 5-fold cross-validation
CONFIG = "3d_fullres"  # Best for KiTS

print(f"\n🚀 Quick test (50 epochs):")
print(f"  nnUNetv2_train {DATASET_ID} {CONFIG} {FOLD} --npz --num_epochs 50")
print(f"\n🏆 Full training (1000 epochs - recommended):")
print(f"  nnUNetv2_train {DATASET_ID} {CONFIG} {FOLD} --npz")
print("\n⏱️ Full training: 2-5 days on GPU")

In [ ]:
# Uncomment to run training from notebook (quick test)
# !nnUNetv2_train {DATASET_ID} 3d_fullres 0 --npz --num_epochs 50

## 6. Inference

In [ ]:
# Inference on test cases
INPUT_FOLDER = Path("./test_images")
OUTPUT_FOLDER = Path("./predictions")
OUTPUT_FOLDER.mkdir(exist_ok=True)

print("="*50)
print("INFERENCE")
print("="*50)
print(f"\nSingle fold:")
print(f"  nnUNetv2_predict -i {INPUT_FOLDER} -o {OUTPUT_FOLDER} -d {DATASET_ID} -c 3d_fullres -f 0")
print(f"\nEnsemble (best quality):")
print(f"  nnUNetv2_predict -i {INPUT_FOLDER} -o {OUTPUT_FOLDER} -d {DATASET_ID} -c 3d_fullres -f all")

## 7. Evaluation & Visualization

In [ ]:
def dice_score(pred, gt, label):
    """Compute Dice score"""
    p = (pred == label).astype(float)
    g = (gt == label).astype(float)
    if p.sum() + g.sum() == 0:
        return 1.0
    return 2 * (p * g).sum() / (p.sum() + g.sum())

def evaluate(pred_path, gt_path):
    """Evaluate prediction"""
    pred = nib.load(pred_path).get_fdata()
    gt = nib.load(gt_path).get_fdata()
    
    return {
        'kidney_dice': dice_score(pred, gt, 1),
        'tumor_dice': dice_score(pred, gt, 2),
        'cyst_dice': dice_score(pred, gt, 3)
    }

In [ ]:
def visualize(img_path, seg_path, slice_idx=None):
    """Visualize CT with segmentation overlay"""
    img = nib.load(img_path).get_fdata()
    seg = nib.load(seg_path).get_fdata()
    
    if slice_idx is None:
        # Find slice with max tumor
        slice_idx = np.argmax(np.sum(seg == 2, axis=(0, 1)))
    
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    
    ax[0].imshow(img[:, :, slice_idx].T, cmap='gray', origin='lower')
    ax[0].set_title('CT'); ax[0].axis('off')
    
    ax[1].imshow(seg[:, :, slice_idx].T, cmap='viridis', origin='lower')
    ax[1].set_title('Segmentation'); ax[1].axis('off')
    
    # Overlay
    ax[2].imshow(img[:, :, slice_idx].T, cmap='gray', origin='lower')
    s = seg[:, :, slice_idx]
    overlay = np.zeros((*s.shape, 4))
    overlay[s == 1] = [0, 1, 0, 0.3]  # Kidney=green
    overlay[s == 2] = [1, 0, 0, 0.5]  # Tumor=red
    overlay[s == 3] = [0, 0, 1, 0.3]  # Cyst=blue
    ax[2].imshow(np.transpose(overlay, (1, 0, 2)), origin='lower')
    ax[2].set_title('Overlay'); ax[2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize sample (uncomment when imaging available)
# visualize(KITS23_DIR / 'case_00000' / 'imaging.nii.gz',
#           KITS23_DIR / 'case_00000' / 'segmentation.nii.gz')

## 8. Quick Reference

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║                  nnU-Net QUICK REFERENCE                      ║
╠══════════════════════════════════════════════════════════════╣
║ 1. SET ENVIRONMENT (PowerShell):                             ║
║    $env:nnUNet_raw = "{}"                    ║
║    $env:nnUNet_preprocessed = "{}"           ║
║    $env:nnUNet_results = "{}"                ║
║                                                              ║
║ 2. PREPROCESS:                                               ║
║    nnUNetv2_plan_and_preprocess -d {} --verify_dataset_integrity ║
║                                                              ║
║ 3. TRAIN (quick test):                                       ║
║    nnUNetv2_train {} 3d_fullres 0 --npz --num_epochs 50      ║
║                                                              ║
║ 4. TRAIN (full):                                             ║
║    nnUNetv2_train {} 3d_fullres 0 --npz                      ║
║                                                              ║
║ 5. PREDICT:                                                  ║
║    nnUNetv2_predict -i INPUT -o OUTPUT -d {} -c 3d_fullres -f 0 ║
╚══════════════════════════════════════════════════════════════╝

Expected Results:
  - Kidney Dice: ~97%
  - Tumor Dice: ~85-90%  
  - Composite: ~91%
""".format(
    NNUNET_RAW.absolute(),
    NNUNET_PREPROCESSED.absolute(), 
    NNUNET_RESULTS.absolute(),
    DATASET_ID, DATASET_ID, DATASET_ID, DATASET_ID
))